# Scenario 5: Continuous Integration Pipeline Reviewer
## Hands-on Production Scenario Workbench
**Certification Focus**: CCAR-F Exam Scenarios 1–6 | **Standard**: Claude 4.5 & Extended Thinking

---

### 📊 Production SLAs & Non-Negotiables
- Headless non-interactive execution via claude -p / --print
- Structured JSON output via --output-format json and --json-schema
- Message Batches API (50% token discount) for non-blocking overnight jobs
- Synchronous API (<3 min turnaround) for blocking pre-merge PR checks
- Two-pass review architecture (per-file local pass + cross-file contract pass)

---

### 🏗️ Architecture & Decision Flow
```mermaid
flowchart TD
    Trigger["CI PR Trigger"] --> WorkloadCheck{"Blocking PR vs Overnight Scan?"}
    
    WorkloadCheck -->|"Pre-merge PR (Need <3 min SLA)"| SyncAPI["Real-time Messages API<br>(claude-sonnet-4-5)"]
    WorkloadCheck -->|"Nightly Full Audit (24h SLA)"| BatchAPI["Message Batches API<br>(50% token discount, custom_id)"]
    
    SyncAPI --> TwoPass["Two-Pass Review Engine"]
    TwoPass --> Pass1["Pass 1: File-Level Syntax & Mocks"]
    TwoPass --> Pass2["Pass 2: Cross-File Integration Contracts"]
    
    Pass2 --> Gate{"Violations Detected?"}
    Gate -->|"High/Critical Violation"| FailPR["Exit Code 1 (Fail CI)"]
    Gate -->|"Clean / Low"| PassPR["Exit Code 0 (Approve PR)"]
```

---

### 🏆 Golden Architectural Rules Reference
| Rule | Production Implementation & Rationale |
|---|---|
| **Headless CLI Flags (TS 3.6)** | Use -p, --output-format json, and --json-schema in automated runners. |
| **Batch API vs Real-Time (TS 4.5)** | Choose Message Batches for overnight jobs (50% discount); real-time API for blocking PRs. |
| **Two-Pass Review (TS 4.6)** | Pass 1 checks file-level logic; Pass 2 checks cross-module interface contracts. |
| **Negative Boundary Criteria (TS 4.1)** | Exclude test mocks, fixtures, and staging credentials from secret detection. |
| **Deterministic Exit Codes (TS 3.6)** | Exit 1 blocks PR merge on critical security/contract flaws; Exit 0 approves. |

---

### 🧠 Claude 4.5 Extended Thinking & Parameter Standards
- **Model Standard**: Use `claude-sonnet-4-5` or `claude-haiku-4-5`.
- **Temperature Omission**: When `thinking` is enabled (`thinking={"type": "enabled", "budget_tokens": ...}`), **do not provide `temperature`**, `top_p`, or `top_k`. The Anthropic API rejects custom sampling values with an HTTP `400 Bad Request` error.
- **Thinking Continuity**: In multi-turn loops, assistant reasoning blocks (`type: "thinking"`) must be returned **unmodified with their cryptographic signature** in subsequent turns.
- **Case Facts Preservation**: Immutable transactional entities must be maintained in a dedicated Case Facts block outside conversation summaries.



In [ ]:
# Step 1: Environment Setup & Simulation Import
import sys
from pathlib import Path

repo_root = Path.cwd()
while repo_root != repo_root.parent and not (repo_root / "pyproject.toml").exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from scenarios.scenario_5_continuous_integration.simulation import *

print(' Scenario Workbench Initialized: Scenario 5: Continuous Integration Pipeline Reviewer')
print(' Zero live API calls, 100% offline determinism')


### 🧪 Step 2: Core Execution Flow & Happy Path Simulation
Demonstrating end-to-end pipeline execution with audit logging.


In [ ]:
# Test Workload API Routing (Synchronous PR vs Message Batches)
engine = ContinuousIntegrationEngine(headless_flag_present=True)

api_pr = engine.route_api_for_workload(WorkloadType.BLOCKING_PR_CHECK)
api_audit = engine.route_api_for_workload(WorkloadType.OVERNIGHT_REPO_AUDIT)
print("Blocking PR API:", api_pr)
print("Overnight Audit API:", api_audit)
assert api_pr == "REALTIME_MESSAGES"
assert api_audit == "MESSAGE_BATCHES"



### ⚠️ Step 3: Edge Case Testing & Guardrail Verifications
Testing boundary conditions, safety gates, error recovery, and escalation protocols.


In [ ]:
# Test Two-Pass Review Pipeline & Negative Boundary Filtering
engine = ContinuousIntegrationEngine(headless_flag_present=True)
pr_files = {
    "api/routes.py": "def handle(bearer_token: str): pass",
    "client/sdk.py": "def call(): return handle(auth_token='abc')",
    "tests/test_app.py": "mock_secret = 'sk-ant-live_fake_in_tests'"
}
report = engine.run_two_pass_review(pr_files)
print("CI Verdict:", report.status)
print("Exit Code:", report.exit_code)
assert report.exit_code == 1
assert report.status == "CHANGES_REQUESTED"
assert any(f.pass_number == 2 for f in report.findings)
assert not any("mock_secret" in f.description for f in report.findings)



### ✅ Step 4: Verification Summary & SLA Compliance
All assertions passed with zero unauthorized actions and 100% deterministic compliance.
